# TP01: the perceptron in NumPy

**MSA-DATB09-03 · Neural Networks & Deep Learning Foundations · Session 1 / 8** · Guillaume Saint-Cirgue

*Estimated time: 70 minutes (Parts A to C after the break, Part D after the course on the limits)*

In the course we computed a perceptron by hand: a score $z = w^\top x + b$, a prediction $\hat y = 1$ if
$z \ge 0$ (else 0), and the learning rule $w \leftarrow w + \eta\,(y - \hat y)\,x$,
$b \leftarrow b + \eta\,(y - \hat y)$. In this notebook you code all of it with NumPy, you check that your
code gives exactly the table of exercise 3, you train it on a real 2-D dataset, and you watch it fail on XOR.

**How it works.** Each exercise is a cell with a zone to complete:

```python
### START CODE HERE ###

### END CODE HERE ###
```

Write your code between the two markers, then run the cell: the `assert` lines below the zone check your
answer and print `Exercise validated!` when it is correct. If an assert fails, read its message: it tells
you what to look at.

**Run it** on Google Colab (File > Upload notebook) or locally with the course `requirements.txt`.

## Imports

- `numpy`: arrays and vectorised maths. The whole perceptron is written with it.
- `matplotlib`: plots (the decision line, the number of mistakes per epoch).
- `sklearn.datasets.make_blobs`: generates clusters of points in 2-D, our toy dataset in Part C.
- `os`: reads the `FAST_RUN` environment variable (used by the instructor to test the notebook quickly).

In [2]:
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs

FAST_RUN = os.environ.get("FAST_RUN", "0") == "1"   # True: quick check run (nothing is long here anyway)
np.random.seed(0)

Matplotlib is building the font cache; this may take a moment.


## Helper functions (run this cell, no need to modify it)

`plot_line` draws a dataset (class 1 as blue dots, class 0 as orange squares) and the line
$w_1 x_1 + w_2 x_2 + b = 0$. `plot_mistakes` draws the number of mistakes per epoch.

In [3]:
def plot_line(X, y, w, b, title=""):
    plt.figure(figsize=(5, 4.5))
    plt.scatter(X[y == 1, 0], X[y == 1, 1], c="#4338ca", label="class 1")
    plt.scatter(X[y == 0, 0], X[y == 0, 1], c="#d97706", marker="s", label="class 0")
    x1 = np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 10)
    if abs(w[1]) > 1e-12:
        plt.plot(x1, -(w[0] * x1 + b) / w[1], "k-", label="decision line")
    plt.ylim(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5)
    plt.xlabel("$x_1$"); plt.ylabel("$x_2$"); plt.title(title); plt.legend()
    plt.show()


def plot_mistakes(mistakes, title=""):
    plt.figure(figsize=(5, 3))
    plt.plot(range(1, len(mistakes) + 1), mistakes, "o-")
    plt.xlabel("epoch"); plt.ylabel("mistakes"); plt.title(title)
    plt.show()

# Part A: predictions

### Exercise 1: the step function

Write `step(z)` that returns 1 where $z \ge 0$ and 0 where $z < 0$. It must work on a single number **and**
on a NumPy array (element by element).

- `z >= 0` compares element by element and gives booleans (`True`/`False`), with the same shape as `z`.
- `.astype(int)` converts booleans to integers: `True` becomes 1, `False` becomes 0.
- `np.asarray(z)` turns a plain Python number into a 0-dimensional NumPy array, so that `.astype` exists.

> Hint: compare with `z >= 0`, then convert with `.astype(int)`; wrap the comparison in `np.asarray(...)` so
> that it also works on a plain number.

In [5]:
def step(z):
    ### START CODE HERE ###

    out = np.asarray(z >= 0).astype(int)

    ### END CODE HERE ###
    return out


assert step(0.5) == 1 and step(-0.5) == 0, "step(0.5) should be 1 and step(-0.5) should be 0"
assert step(0) == 1, "careful at z = 0: the course convention gives 1"
assert np.array_equal(step(np.array([-2.0, 0.0, 3.0])), [0, 1, 1]), "step must work element by element on an array"
print("Exercise validated!")

Exercise validated!


### Exercise 2: the prediction for one example

Write `predict_one(x, w, b)` for one example `x` (shape `(n,)`), weights `w` (shape `(n,)`) and a bias `b`:
compute the score $z = w^\top x + b$, then return `step(z)`.

- `w @ x` is the dot product $\sum_j w_j x_j$ of two 1-D arrays (you can also write `np.dot(w, x)`).

We test it on exercise 1 of the course: $w = (1, 1)$, $b = -1.5$ computes AND.

In [6]:
def predict_one(x, w, b):
    ### START CODE HERE ###
    y_hat = step(np.dot(w, x) + b)
    ### END CODE HERE ###
    return y_hat


w_and, b_and = np.array([1.0, 1.0]), -1.5
assert predict_one(np.array([1, 1]), w_and, b_and) == 1, "AND(1, 1) should be 1: z = 1 + 1 - 1.5 = 0.5"
assert predict_one(np.array([0, 1]), w_and, b_and) == 0, "AND(0, 1) should be 0: z = 0 + 1 - 1.5 = -0.5"
print("Exercise validated!")

Exercise validated!


### Exercise 3: the truth tables as arrays

Create the four inputs of a truth table as a matrix `X_table` of shape `(4, 2)`, one row per example, in
the order (0, 0), (0, 1), (1, 0), (1, 1). Then create the label vectors `y_and`, `y_or` and `y_xor`
(shape `(4,)`).

- `np.array([[a, b], [c, d]])` builds a 2-D array from a list of rows.

In [8]:
### START CODE HERE ###

X_table = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_and = np.array([0, 0, 0, 1])
y_or = np.array([0, 1, 1, 1])
y_xor = np.array([0, 1, 1, 0])

### END CODE HERE ###

assert X_table.shape == (4, 2), f"X_table should have shape (4, 2), not {np.shape(X_table)}"
assert X_table[3].tolist() == [1, 1], "the last row should be (1, 1)"
assert y_and.tolist() == [0, 0, 0, 1] and y_or.tolist() == [0, 1, 1, 1], "check the AND and OR labels"
assert y_xor.tolist() == [0, 1, 1, 0], "XOR is 1 when exactly one input is 1"
print("Exercise validated!")

Exercise validated!


### Exercise 4: vectorised predictions

Write `predict(X, w, b)` that returns the predictions of **all** the rows of `X` at once, without a Python
loop.

- `X @ w` with `X` of shape `(m, n)` and `w` of shape `(n,)` gives the vector of the $m$ dot products, shape
  `(m,)`. Adding the number `b` adds it to every element (this is called broadcasting).

> Hint: one line, `step(X @ w + b)`. The same function also works for one example of shape `(n,)`.

In [ ]:
def predict(X, w, b):
    ### START CODE HERE ###
    y_hat = step(np.dot(X, w) + b)
    ### END CODE HERE ###
    return y_hat

assert predict(X_table, w_and, b_and).shape == (4,), "predict should return one prediction per row"
assert np.array_equal(predict(X_table, w_and, b_and), y_and), "w = (1, 1), b = -1.5 should compute AND"
print("Exercise validated!")

Exercise validated!


### Exercise 5: weights for OR and NAND

Now you choose the parameters, as in exercise 1 of the course. Set `w_or`, `b_or` so that the perceptron
computes OR, and `w_nand`, `b_nand` so that it computes NAND (NOT AND: 1, 1, 1, 0). Keep these NAND
weights in mind: they come back in session 3.

> Hint: for OR keep $w = (1, 1)$ and move the line with $b$. For NAND, flip all the signs of AND.

In [ ]:
### START CODE HERE ###

### END CODE HERE ###

assert np.array_equal(predict(X_table, w_or, b_or), y_or), "your OR perceptron is wrong on at least one input"
assert np.array_equal(predict(X_table, w_nand, b_nand), [1, 1, 1, 0]), "your NAND perceptron is wrong"
print("Exercise validated!")

# Part B: the learning rule

### Exercise 6: one update

Write `update(x, y, w, b, eta)` that applies the perceptron rule to **one** example and returns the new
`w` and `b`:

$$ e = y - \hat y, \qquad w \leftarrow w + \eta\, e\, x, \qquad b \leftarrow b + \eta\, e $$

Do not modify `w` in place (no `w += ...`): build a new array with `w + ...`, so that the caller's array is
not changed behind its back.

In [ ]:
def update(x, y, w, b, eta=1.0):
    ### START CODE HERE ###
    error = None
    ### END CODE HERE ###
    return w, b


w0, b0 = np.zeros(2), 0.0
w1, b1 = update(np.array([1.0, 0.0]), 0, w0, b0)          # point B of exercise 3: false positive
assert np.allclose(w1, [-1, 0]) and b1 == -1, f"false positive: expected w = (-1, 0), b = -1, got {w1}, {b1}"
w2, b2 = update(np.array([3.0, 1.0]), 1, w1, b1)          # point C: false negative
assert np.allclose(w2, [2, 1]) and b2 == 0, f"false negative: expected w = (2, 1), b = 0, got {w2}, {b2}"
w3, b3 = update(np.array([1.0, 3.0]), 1, w2, b2)          # correct prediction: nothing changes
assert np.allclose(w3, w2) and b3 == b2, "a correct prediction must not change the parameters"
assert np.allclose(w0, 0), "do not modify w in place"
print("Exercise validated!")

### Exercise 7: count the mistakes

Write `count_mistakes(X, y, w, b)`: the number of examples whose prediction differs from the label.

- `a != b` on two arrays compares element by element; `np.sum` of booleans counts the `True` values.

In [ ]:
def count_mistakes(X, y, w, b):
    ### START CODE HERE ###
    n = None
    ### END CODE HERE ###
    return n


assert count_mistakes(X_table, y_and, w_and, b_and) == 0, "the AND perceptron makes no mistake on AND"
assert count_mistakes(X_table, y_or, w_and, b_and) == 2, "the AND perceptron makes 2 mistakes on OR"
print("Exercise validated!")

### Exercise 8: the training loop

Write `train_perceptron(X, y, eta, n_epochs)`:

1. start from `w = np.zeros(n)` (with `n = X.shape[1]`, the number of features) and `b = 0.0`;
2. for each epoch, visit the examples **in order** (`for x_i, y_i in zip(X, y):`), apply `update`, and count
   the mistakes made **during** the epoch (an update happened when the prediction was wrong);
3. append this count to the list `mistakes`; stop early (`break`) when an epoch has no mistake;
4. return `w, b, mistakes`.

- `zip(X, y)` walks through the rows of `X` and the elements of `y` together.

> Hint: before calling `update`, test `predict(x_i, w, b) != y_i` to know if this example is a mistake.

In [ ]:
def train_perceptron(X, y, eta=1.0, n_epochs=20):
    ### START CODE HERE ###
    w, b, mistakes = None, None, None
    ### END CODE HERE ###
    return w, b, mistakes


w, b, mistakes = train_perceptron(X_table, y_and)
print("AND: w =", w, " b =", b, " mistakes per epoch:", mistakes)
assert mistakes[-1] == 0, "on AND (separable) the last epoch should have no mistake"
assert np.array_equal(predict(X_table, w, b), y_and), "the trained perceptron should compute AND"
print("Exercise validated!")

### Exercise 9: OR

Train a perceptron on the OR table: store the results in `w_or_learnt`, `b_or_learnt`, `mistakes_or`.

In [ ]:
### START CODE HERE ###

### END CODE HERE ###

print("OR: w =", w_or_learnt, " b =", b_or_learnt, " mistakes per epoch:", mistakes_or)
assert mistakes_or[-1] == 0, "OR is separable: the last epoch should have no mistake"
assert np.array_equal(predict(X_table, w_or_learnt, b_or_learnt), y_or), "the trained perceptron should compute OR"
print("Exercise validated!")

### Exercise 10: exercise 3 of the course, exactly

The four points A = (1, 3), B = (1, 0), C = (3, 1), D = (0, 0) with labels 1, 0, 1, 0, visited in this
order, from $w = (0, 0)$, $b = 0$, $\eta = 1$. Create `X_ex3` and `y_ex3`, train, and store the results in
`w_ex3, b_ex3, mistakes_ex3`. Compare with your table on paper: the asserts check the course solution.

In [ ]:
### START CODE HERE ###

### END CODE HERE ###

print("w =", w_ex3, " b =", b_ex3, " mistakes per epoch:", mistakes_ex3)
assert mistakes_ex3 == [3, 1, 0], f"expected 3, 1, 0 mistakes per epoch, got {mistakes_ex3}"
assert np.allclose(w_ex3, [1, 1]) and b_ex3 == -2, "expected the final line x1 + x2 - 2 = 0"
print("Exercise validated!")
plot_line(X_ex3, y_ex3, w_ex3, b_ex3, "Exercise 3: the final line")

# Part C: a 2-D dataset

### Exercise 11: generate and train

`make_blobs` draws points around given centres:

- `n_samples=100`: 100 points in total, shared between the clusters;
- `centers=[[-1.5, -1.5], [1.5, 1.5]]`: two clusters, centred at $(-1.5, -1.5)$ and $(1.5, 1.5)$;
- `cluster_std=1.0`: the spread of each cluster (the standard deviation);
- `random_state=3`: fixes the random draw, so everybody gets the same points.

It returns `X` of shape `(100, 2)` and the cluster index `y` (0 or 1) of shape `(100,)`.
Train a perceptron on it for at most 20 epochs: `w_blobs, b_blobs, mistakes_blobs`.

In [ ]:
X_blobs, y_blobs = make_blobs(n_samples=100, centers=[[-1.5, -1.5], [1.5, 1.5]], cluster_std=1.0, random_state=3)

### START CODE HERE ###

### END CODE HERE ###

print("w =", w_blobs, " b =", b_blobs, " mistakes per epoch:", mistakes_blobs)
assert X_blobs.shape == (100, 2), "do not change the dataset"
assert mistakes_blobs[-1] == 0, "the two clusters are separable: training should end with an epoch without mistake"
print("Exercise validated!")
plot_mistakes(mistakes_blobs, "Two separable clusters")

### Exercise 12: accuracy

Write `accuracy(y_true, y_pred)`: the fraction of correct predictions, a number between 0 and 1.

- `np.mean` of a boolean array is the fraction of `True` values.

In [ ]:
def accuracy(y_true, y_pred):
    ### START CODE HERE ###
    acc = None
    ### END CODE HERE ###
    return acc


assert accuracy(np.array([1, 0, 1, 1]), np.array([1, 1, 1, 0])) == 0.5, "2 correct out of 4 should give 0.5"
acc_blobs = accuracy(y_blobs, predict(X_blobs, w_blobs, b_blobs))
print("training accuracy:", acc_blobs)
assert acc_blobs == 1.0, "on separable data the trained perceptron classifies every training point correctly"
print("Exercise validated!")

### Exercise 13: the equation of the line

To draw the line $w_1 x_1 + w_2 x_2 + b = 0$, we solve it for $x_2$:

$$ x_2 = -\frac{w_1 x_1 + b}{w_2} \qquad (w_2 \neq 0) $$

Write `line_x2(x1, w, b)` that returns $x_2$ for a number or an array `x1`. Then check that a point on the
line has a score of 0.

In [ ]:
def line_x2(x1, w, b):
    ### START CODE HERE ###
    x2 = None
    ### END CODE HERE ###
    return x2


assert np.isclose(line_x2(0.0, np.array([1.0, 1.0]), -2.0), 2.0), "the line x1 + x2 - 2 = 0 crosses x1 = 0 at x2 = 2"
x1_test = np.array([-1.0, 0.5, 3.0])
z_on_line = w_blobs[0] * x1_test + w_blobs[1] * line_x2(x1_test, w_blobs, b_blobs) + b_blobs
assert np.allclose(z_on_line, 0), "points (x1, line_x2(x1)) must have a score of 0"
print("Exercise validated!")
plot_line(X_blobs, y_blobs, w_blobs, b_blobs, "Perceptron on two clusters")

### Exercise 14: does the learning rate matter?

Train again on the clusters with `eta=0.1` and store `w_small, b_small, mistakes_small`. Compare with
`eta=1`: the asserts check two surprising facts. Read the "What we see" cell after it.

In [ ]:
### START CODE HERE ###

### END CODE HERE ###

assert np.allclose(w_small * 10, w_blobs) and np.isclose(b_small * 10, b_blobs), "with eta = 0.1 the weights are 10 times smaller"
assert mistakes_small == mistakes_blobs, "the mistakes per epoch are exactly the same"
print("Exercise validated!")

**What we see.** Starting from $w = 0$, every update adds $\eta$ times something, so all the parameters are
multiplied by $\eta$. The score $z$ is multiplied by $\eta$ too, and the step only looks at the **sign** of
$z$: the predictions, hence the whole training, are identical. For the perceptron the learning rate does
not matter. It will matter a lot with gradient descent (session 2).

# Part D: the limits

### Exercise 15: XOR

Train on the XOR table for 50 epochs: `w_xor, b_xor, mistakes_xor`. Exercise 4 of the course proved that
no line works: the asserts check that the rule never finds an epoch without mistake.

In [ ]:
### START CODE HERE ###

### END CODE HERE ###

print("mistakes per epoch (first 10):", mistakes_xor[:10])
assert len(mistakes_xor) == 50, "training should use all 50 epochs (it never stops early)"
assert min(mistakes_xor) > 0, "no epoch can be perfect on XOR"
assert count_mistakes(X_table, y_xor, w_xor, b_xor) >= 1, "the final line is wrong on at least one XOR input"
print("Exercise validated!")

### Exercise 16: overlapping clusters

Same clusters as Part C, but much more spread (`cluster_std=2.0`): they overlap. Train for 50 epochs:
`w_over, b_over, mistakes_over`, then compute the training accuracy `acc_over`.

In [ ]:
X_over, y_over = make_blobs(n_samples=100, centers=[[-1.5, -1.5], [1.5, 1.5]], cluster_std=2.0, random_state=3)

### START CODE HERE ###

### END CODE HERE ###

print("mistakes per epoch (first 10):", mistakes_over[:10], " final accuracy:", acc_over)
assert len(mistakes_over) == 50 and min(mistakes_over) > 0, "on overlapping data every epoch has mistakes"
assert 0.5 < acc_over < 1.0, "the accuracy is good but cannot be perfect"
print("Exercise validated!")
plot_mistakes(mistakes_over, "Overlapping clusters: it never stops")
plot_line(X_over, y_over, w_over, b_over, "Overlapping clusters: the line after 50 epochs")

**What we see.** The number of mistakes goes up and down forever: each correction fixes one point and
breaks another. The final line is simply the one we have when we stop; train for 49 or 51 epochs and you
get another one. Nothing tells the perceptron which line is the "best".

### Exercise 17: the step function has no slope

Gradient descent needs the derivative of the output with respect to $z$. Estimate it numerically with a
centred finite difference:

$$ \text{slope}(z) \approx \frac{\text{step}(z + h) - \text{step}(z - h)}{2h}, \qquad h = 10^{-3} $$

Compute `slopes`, the array of these estimates at `z_values` (use `step`, it works on arrays).

In [ ]:
z_values = np.array([-3.0, -1.0, -0.1, 0.1, 1.0, 3.0])
h = 1e-3

### START CODE HERE ###

### END CODE HERE ###

print("slopes:", slopes)
assert np.shape(slopes) == (6,), "one slope per value of z"
assert np.allclose(slopes, 0), "the step is flat everywhere except at 0: all slopes are 0"
print("Exercise validated!")

### Exercise 18: the four limits, in your own words

Write the four limits of the perceptron seen in the course, one sentence each, and for each one the fix
that the next sessions bring.

> *Your answer here.*

# Going further

### Exercise 19: a perceptron class, scikit-learn style

scikit-learn models are classes with a `fit(X, y)` method that learns and a `predict(X)` method that
predicts. Complete the class: `fit` must store the learnt parameters in `self.w`, `self.b` and the list
`self.mistakes` (reuse `train_perceptron`), and return `self`; `predict` must reuse the function `predict`.

In [ ]:
class Perceptron:
    def __init__(self, eta=1.0, n_epochs=20):
        self.eta = eta                    # learning rate
        self.n_epochs = n_epochs          # maximum number of epochs

    def fit(self, X, y):
        ### START CODE HERE ###
        self.w, self.b, self.mistakes = None, None, None
        ### END CODE HERE ###
        return self

    def predict(self, X):
        ### START CODE HERE ###
        y_hat = None
        ### END CODE HERE ###
        return y_hat


model = Perceptron().fit(X_blobs, y_blobs)
assert np.allclose(model.w, w_blobs) and np.isclose(model.b, b_blobs), "fit should give the weights of train_perceptron"
assert accuracy(y_blobs, model.predict(X_blobs)) == 1.0, "predict should classify the clusters perfectly"
print("Exercise validated!")

### Exercise 20: compare with scikit-learn

`sklearn.linear_model.Perceptron` implements the same rule. Its main arguments: `eta0` (the learning rate),
`max_iter` (the number of epochs), `tol=None` (otherwise it may stop early, as soon as its internal loss
stops improving), `random_state` (it shuffles the examples at each epoch).
After `fit`, the weights are in `coef_` (shape `(1, n)`) and the bias in `intercept_` (shape `(1,)`).
Fit it on the clusters (`eta0=1.0, max_iter=20, tol=None, random_state=0`) as `sk_model` and compute its training
accuracy `acc_sk`.

In [ ]:
from sklearn.linear_model import Perceptron as SkPerceptron

### START CODE HERE ###

### END CODE HERE ###

print("scikit-learn: w =", sk_model.coef_[0], " b =", sk_model.intercept_[0], " accuracy:", acc_sk)
assert sk_model.coef_.shape == (1, 2), "coef_ should have shape (1, 2)"
assert acc_sk == 1.0, "scikit-learn's perceptron also separates the clusters"
print("Exercise validated!")

**What we see.** Same accuracy, different weights: scikit-learn visits the examples in a random order at
each epoch, and many lines separate two well-separated clusters. The perceptron finds *a* separating line,
not a particular one.